In [32]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/MaliciousCallDetection/

!cp /content/drive/MyDrive/MaliciousCallDetection/dataset/spectograms_backup.zip /content/

!unzip /content/spectograms_backup.zip -d /content/spectograms


!git config --global user.email "marinescudragos2014@gmail.com"
!git config --global user.name "Marinescu Dragos"


In [ ]:
!pip install -r /content/drive/MyDrive/MaliciousCallDetection/requirements.txt

In [ ]:
import wandb
wandb.login()

In [ ]:
from collections import Counter
from pathlib import Path

folder = Path("/content/drive/MyDrive/MaliciousCallDetection/dataset/normal")

extensions = Counter(
    p.suffix.lower() or "<no extension>"
    for p in folder.iterdir()
    if p.is_file()
)

extensions

Counter({'.wav': 1648, '.mp3': 40})

In [29]:

def save_spectograms(base_path, output_path, sr=16000, window_size=5.0, min_overlap=1.5):
    window_size_samples = int(window_size * sr)
    min_overlap_samples = int(min_overlap * sr)

    for category in ['normal', 'malicious']:
        os.makedirs(os.path.join(output_path, category), exist_ok=True)

    print(f"Processing data from '{base_path}'...")

    for label, category in enumerate(['normal', 'malicious']):
        dir_path = os.path.join(base_path, category)
        if not os.path.exists(dir_path):
            continue

        valid_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.wav', '.mp3'))]

        print(f"\nProcessing category: {category.upper()} ({len(valid_files)} files)")

        for f in tqdm(valid_files, desc=f"{category} progress", unit="file"):
            path = os.path.join(dir_path, f)

            try:
                audio_full, _ = librosa.load(path, sr=sr)
                total_samples = len(audio_full)
            except Exception as e:
                print(f"Error loading {f}: {e}")
                continue

            if total_samples < int(8.5 * sr) and total_samples >= window_size_samples:
                step = max(1, total_samples - window_size_samples)
            else:
                step = window_size_samples - min_overlap_samples

            window_idx = 0
            for start in range(0, total_samples - window_size_samples + 1, step):
                save_file_name = f"{os.path.splitext(f)[0]}_spectogram_{window_idx}.pt"
                save_path = os.path.join(output_path, category, save_file_name)

                if os.path.exists(save_path):
                    window_idx += 1
                    continue

                # in-memory slicing
                audio_segment = audio_full[start : start + window_size_samples]

                mel_spec = librosa.feature.melspectrogram(y=audio_segment, sr=sr, n_mels=128)
                log_mel = librosa.power_to_db(mel_spec, ref=np.max)
                tensor_mel = torch.tensor(log_mel).unsqueeze(0)

                torch.save(tensor_mel, save_path)
                window_idx += 1

    print(f"\nDone! Every spectogram is in '{output_path}'.")
save_spectograms("dataset","dataset/spectograms")

Processing data from 'dataset'...

Done! Every spectogram is in 'dataset/spectograms'.


In [ ]:
%cd ../malicious/ #am cun 6000 mai multe spectograme malitioase decat normale
%ls -1 | wc -l

/content/drive/MyDrive/MaliciousCallDetection/dataset/spectograms/malicious
28014


In [15]:

class Spectrogram(Dataset):
  def __init__(self,spectogram_path):
    self.samples=[]

    for label, category in enumerate(['normal', 'malicious']):
      dir_path = os.path.join(spectogram_path, category)
      for f in os.listdir(dir_path):
        if f.endswith('.pt'):
          self.samples.append((os.path.join(dir_path,f),label))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    path,label=self.samples[idx]
    mel_tensor=torch.load(path)
    return mel_tensor,label

p=Spectrogram('/content/drive/MyDrive/MaliciousCallDetection/dataset/spectograms')

# fig, axes = plt.subplots(2, 5, figsize=(20, 8))
# axes = axes.flatten()

# random_spectograms=np.random.choice(len(p), size=10, replace=False)
# for idx,spectogram_idx in enumerate(random_spectograms):
#     mel, label = p[spectogram_idx]          # mel: (1, 128, 157)
#     mel = mel.squeeze(0)       # (128, 157)

#     axes[idx].imshow(
#         mel,
#         origin='lower',
#         aspect='auto',
#         cmap='magma'
#     )

#     axes[idx].set_title(f"Label: {'malicious' if label==1 else 'normal'}")
#     axes[idx].set_xlabel("Time")
#     axes[idx].set_ylabel("Mel bins")

# plt.tight_layout()
# plt.show()

NameError: name 'Dataset' is not defined

In [21]:

def configure_optimizer(model: nn.Module,lr=0.001) -> optim.Optimizer:
    return optim.Adam(model.parameters(), lr)

class CNN(nn.Module): #Dropout , BatchNorm?
    def __init__(self):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            #we use padding because we don't want to loose pixels from the images margins, now the output matrix is exactly as the input
            #in spectograms this is important
            #formula : out=(W−K+2P)/S ​+ 1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),#in_channels=1 because spectogram is grayscale not RGB
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),#reduce image in half, concentrating on the most imp pixel

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Block 2: The "Bulletproof" Funnel
        # This forces the spatial dimensions down to 1x1, regardless of input length.
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Block 3: Classification
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 128), # It's 32 because the last Conv2d output 32 channels
            nn.ReLU(),
            nn.Dropout(0.5), # Added dropout to prevent overfitting
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x



In [22]:
def evaluate_best_model(model_path: str,
                        test_loader: torch.utils.data.DataLoader,
                        device: torch.device,
                        run: Optional[wandb.sdk.wandb_run.Run]):

  if not os.path.exists(model_path):
    print(f"ERORRE: Nu am găsit niciun model la calea: {model_path}")
    return

  print(f"Load the model from: {model_path}...")

  model = CNN().to(device)
  model.load_state_dict(torch.load(model_path, map_location=device))

  model.eval()

  all_preds = []
  all_labels = []

  print("Start testing...")

  with torch.no_grad():
    for X_test, y_test in test_loader:
      X_test = X_test.to(device)
      y_test = y_test.to(device)

      pred = model(X_test)
      predicted=pred.argmax(dim=1)

      # GPU->CPU for Scikit-Learn
      all_preds.extend(predicted.cpu().numpy())
      all_labels.extend(y_test.cpu().numpy())

  accuracy = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
  precision = precision_score(all_labels, all_preds, zero_division=0) #How many relevant samples did we get  tp/(tp+fp)
  recall = recall_score(all_labels, all_preds, zero_division=0) #How many malicious did we get right tp/(tp+fn)
  f1 = f1_score(all_labels, all_preds, zero_division=0) #Final grade  2*p*r/(p+r)
  if run:
    run.summary["test/accuracy"] = accuracy
    run.summary["test/precision"] = precision
    run.summary["test/recall"] = recall
    run.summary["test/f1"] = f1

    run.log({"test/confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=['Normal', 'Malicious']
        )})


  print("\n" + "="*40)
  print("        FINAL RESULTS")
  print("="*40)
  print(f"Accuracy: {accuracy:.2f}%")
  print(f"Precision: {precision:.4f} How many relevant samples did we get")
  print(f"Sensibilitate (Recall): {recall:.4f} How many malicious did we get right")
  print(f"F1-Score: {f1:.4f} Final Grade")

  # Confusion Matrix
  cm = confusion_matrix(all_labels, all_preds)

  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
              xticklabels=['Normal', 'Malicious'],
              yticklabels=['Normal', 'Malicious'],
              annot_kws={"size": 16})

  plt.ylabel('Real', fontsize=14)
  plt.xlabel('Predicted', fontsize=14)
  plt.title('Confusion Matrix', fontsize=16)
  plt.tight_layout()
  plt.show()

In [18]:
def train_model(path: str,
                model_no: int,
                lr: float,
                epochs: int,
                bs: int):

  model = CNN()
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = model.to(device)
  if device.type=="cpu" :
    print("ERROR! Not using GPU")
    return

  run=wandb.init(entity="dragosmarinescu19-babes-bolyai-university",
              name=f"run{model_no}",
              project="MaliciousCallDetection",
              config={
                  "lr":lr,
                  "batch_size":bs,
                  "epochs":epochs,
                  "optimizer":"Adam",
                  "model_architecture": "CNN"
               })
  # weights = torch.tensor([2.0, 1.0]).to(device)
  # loss_func = nn.CrossEntropyLoss(weight=weights)
  loss_func = nn.CrossEntropyLoss()
  optimizer = configure_optimizer(model,run.config.lr)

  dataset = Spectrogram(path)
  train_sz = int(0.8 * len(dataset))
  val_sz = int(0.1 * len(dataset))
  test_sz = len(dataset) - train_sz - val_sz

  train_db, val_db, test_db = random_split(dataset, [train_sz, val_sz, test_sz], generator=torch.Generator().manual_seed(1)) #to avoid Data Leakage

  train_loader = DataLoader(train_db, batch_size=run.config.batch_size, shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
  val_loader = DataLoader(val_db, batch_size=run.config.batch_size, shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
  test_loader = DataLoader(test_db, batch_size=run.config.batch_size, shuffle=False,num_workers=2,pin_memory=True)

  epochs = run.config.epochs

  best_val_loss = float('inf') # We look after the best score
  best_model_path = f'/content/drive/MyDrive/MaliciousCallDetection/models/malicious_call_detector_v{model_no}_best.pt'

  print(f"Starting training on {device}...")

  for epoch in range(epochs):
    model.train() # Put model in training mode
    train_loss = 0.0
    train_correct=0
    train_total=0

    batch_loop=tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=False)

    for X_train, y_train in batch_loop:
      # Move data to the same device as the model
      X_train = X_train.to(device)
      y_train = y_train.to(device)

      optimizer.zero_grad()

      pred = model(X_train)

      loss = loss_func(pred, y_train)

      loss.backward() #calculate how to fix the error

      optimizer.step() #update the weights

      train_loss += loss.item()*X_train.size(0)

      #compare the prediction vector to the y_train vector wich contains labels, then we'll have a boolean tensor
      #after the sum we'll have the number of True elements inside a scalar tensor
      #then we use item() to get the number as a python int
      train_correct+=(pred.argmax(dim=1)==y_train).sum().item()
      train_total+=y_train.size(0)

      current_acc=100 * train_correct / train_total
      batch_loop.set_postfix(loss=loss.item(), acc=f"{current_acc:.2f}%")

    train_acc=train_correct/train_total
    avg_train_loss=train_loss/len(train_loader.dataset)

    # VALIDATION

    model.eval() # STOP the dropout
    val_loss = 0.0
    val_correct=0
    val_total=0

    with torch.no_grad():
      for X_val, y_val in val_loader:
        X_val, y_val = X_val.to(device), y_val.to(device)

        pred = model(X_val)
        loss = loss_func(pred, y_val)
        val_loss += loss.item() * X_val.size(0)

        # Accuracy
        val_correct += (pred.argmax(dim=1) == y_val).sum().item()
        val_total+=y_val.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)
    val_acc = val_correct/val_total
    run.log({"train/loss":avg_train_loss,
             "train/accuracy":train_acc,
             "val/loss":avg_val_loss,
             "val/accuracy":val_acc,
             "epoch":epoch+1
             })

    print(f"Epoch [{epoch+1}/{epochs}] | Loss Train: {avg_train_loss:.4f} | Loss Validation: {avg_val_loss:.4f} | Accuracy: {val_acc*100:.2f}%")

    #SAVE
    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      torch.save(model.state_dict(), best_model_path)
      print(f"Validation Loss droped. We saved the current model.")

  print(f"Done training. Best model saved as: {best_model_path}")

  evaluate_best_model(best_model_path, test_loader, device,run)

  run.finish()


In [19]:
from tqdm import tqdm
from typing import Optional
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import wandb
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import os
import librosa


def main(model_version: int,
         lr: float=3e-4,
         epochs: int=10,
         batch_size: int=32):
  data_path='/content/spectograms/spectograms'


 train_model(
        path=data_path,
        model_no=model_version,
        lr=lr,
        epochs=epochs,
        bs=batch_size
    )

if __name__ == "__main__":
  main(model_version=2)


In [24]:
def audio_to_tensor(file_path, sr=16000, n_mels=128, window_size=5.0):
  audio, _ = librosa.load(file_path, sr=sr)
  window_samples = int(sr * window_size)

  if len(audio) < window_samples:
      audio = np.pad(audio, (0, window_samples - len(audio)))
  else:
      audio = audio[:window_samples]

  mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
  log_mel = librosa.power_to_db(mel_spec, ref=np.max)
  tensor = torch.tensor(log_mel).unsqueeze(0).unsqueeze(0)  # shape: [1, 1, n_mels, time]
  return tensor

def record_audio(filename: str, duration: float = 5.0, sr: int = 16000):

  os.makedirs(os.path.dirname(filename), exist_ok=True)

  print(f"Recording {duration}s of audio...")
  audio = sd.rec(int(duration * sr), samplerate=sr, channels=1, dtype='float32')
  sd.wait()
  write(filename, sr, audio)
  print(f"Saved audio to: {filename}")

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import os
import librosa
import torch
import numpy as np

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN()
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/MaliciousCallDetection/models/malicious_call_detector_v1_best.pt",
    map_location=device
))
model.to(device)
model.eval()

file_path = "/content/drive/MyDrive/MaliciousCallDetection/records/normal_2.wav"
#record_audio(file_path, duration=5)  # I can not record in colab
x = audio_to_tensor(file_path).to(device)

with torch.no_grad():
    pred = model(x)
    predicted_class = pred.argmax(dim=1).item()  # 0 = normal, 1 = malicious
    confidence = torch.softmax(pred, dim=1)[0, predicted_class].item()

print(f"Predicted class: {predicted_class}, confidence: {confidence:.2f}")


In [ ]:
!git add .
!git commit -m 'runned first model. Worked good. I need to randomize the audio files now and not the spectograms.'
!git push

In [ ]:
!git status